In [3]:
import jax
import jax.numpy as jnp
import numpy as np
from jax import lax

jax.config.update("jax_enable_x64", False)

# --- tiny SU(3) core ---
def su3_generators():
    lam = []
    lam.append(jnp.array([[0,1,0],[1,0,0],[0,0,0]], jnp.complex64))
    lam.append(jnp.array([[0,-1j,0],[1j,0,0],[0,0,0]], jnp.complex64))
    lam.append(jnp.array([[1,0,0],[0,-1,0],[0,0,0]], jnp.complex64))
    lam.append(jnp.array([[0,0,1],[0,0,0],[1,0,0]], jnp.complex64))
    lam.append(jnp.array([[0,0,-1j],[0,0,0],[1j,0,0]], jnp.complex64))
    lam.append(jnp.array([[0,0,0],[0,0,1],[0,1,0]], jnp.complex64))
    lam.append(jnp.array([[0,0,0],[0,0,-1j],[0,1j,0]], jnp.complex64))
    lam.append(jnp.array([[1,0,0],[0,1,0],[0,0,-2]], jnp.complex64)/jnp.sqrt(3.0))
    lam = jnp.stack(lam, axis=0)
    return 1j * lam / 2.0

T = su3_generators()

def su3_alg_from_vec(a):
    return jnp.einsum("a,aij->ij", a, T)

def su3_exp_pade22(A):
    I = jnp.eye(3, dtype=jnp.complex64)
    A2 = A @ A
    Num = I + 0.5*A + (1/12)*A2
    Den = I - 0.5*A + (1/12)*A2
    return jnp.linalg.solve(Den, Num)

# ---- tiny L=2 Wilson action ----
def build_links(theta):
    # theta: shape (512,) for L=2
    flat = theta.reshape(-1,8)
    A = jax.vmap(su3_alg_from_vec)(flat)
    U = jax.vmap(su3_exp_pade22)(A)
    return U.reshape(2,2,2,2,4,3,3)

def wilson_action(theta, beta=1.0):
    U = build_links(theta)
    S = 0.0
    for mu in range(4):
        for nu in range(mu+1,4):
            U1 = U[..., mu, :, :]
            U2 = jnp.roll(U[..., nu, :, :], -1, axis=mu)
            U3 = jnp.swapaxes(jnp.conjugate(jnp.roll(U[..., mu, :, :], -1, axis=nu)), -1, -2)
            U4 = jnp.swapaxes(jnp.conjugate(U[..., nu, :, :]), -1, -2)
            P = U1 @ U2 @ U3 @ U4
            S += jnp.sum(1.0 - jnp.real(jnp.einsum("...ii->...", P))/3.0)
    return beta * S

# ---- Hessian-vector product and tiny Lanczos ----
flat_action = jax.jit(wilson_action)

def hvp(theta, v):
    g = jax.grad(flat_action)
    _, hv = jax.jvp(g, (theta,), (v,))
    return hv

def lanczos_min(theta, k=10):
    n = theta.shape[0]
    key = jax.random.PRNGKey(0)
    v0 = jax.random.normal(key, (n,))
    v0 /= jnp.linalg.norm(v0)

    def step(carry,_):
        v_prev, v_cur, beta_prev = carry
        w = hvp(theta, v_cur)
        w -= beta_prev * v_prev
        alpha = jnp.dot(w, v_cur)
        w -= alpha * v_cur
        beta = jnp.linalg.norm(w)
        v_next = w/(beta+1e-9)
        return (v_cur,v_next,beta),(alpha,beta)

    (_,_,_), (a,b) = lax.scan(step, (jnp.zeros_like(v0), v0, 0.0), None, length=k)
    T = jnp.diag(a) + jnp.diag(b[:-1],1) + jnp.diag(b[:-1],-1)
    return float(jnp.linalg.eigvalsh(T)[0])

# ---- run simple test ----
L = 2
n = (L**4)*4*8
theta0 = jnp.zeros((n,), dtype=jnp.float32)

lam0 = lanczos_min(theta0)
print("Lanczos λ_min at θ=0:", lam0)

key = jax.random.PRNGKey(1)
theta_rand = 0.02 * jax.random.normal(key, (n,), dtype=jnp.float32)
lam_rand = lanczos_min(theta_rand)
print("Lanczos λ_min at small θ:", lam_rand)


/usr/local/lib/python3.12/dist-packages/jax/_src/lax/lax.py:5473: ComplexWarning: Casting complex values to real discards the imaginary part
  x_bar = _convert_element_type(x_bar, x.aval.dtype, x.aval.weak_type)


Lanczos λ_min at θ=0: 8.905772119760513e-09
Lanczos λ_min at small θ: -0.011056306771934032


In [4]:
import jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", False)

# SU(3) generators
def su3_generators():
    lam = []
    lam.append(jnp.array([[0,1,0],[1,0,0],[0,0,0]], jnp.complex64))
    lam.append(jnp.array([[0,-1j,0],[1j,0,0],[0,0,0]], jnp.complex64))
    lam.append(jnp.array([[1,0,0],[0,-1,0],[0,0,0]], jnp.complex64))
    lam.append(jnp.array([[0,0,1],[0,0,0],[1,0,0]], jnp.complex64))
    lam.append(jnp.array([[0,0,-1j],[0,0,0],[1j,0,0]], jnp.complex64))
    lam.append(jnp.array([[0,0,0],[0,0,1],[0,1,0]], jnp.complex64))
    lam.append(jnp.array([[0,0,0],[0,0,-1j],[0,1j,0]], jnp.complex64))
    lam.append(jnp.array([[1,0,0],[0,1,0],[0,0,-2]], jnp.complex64)/jnp.sqrt(3))
    lam = jnp.stack(lam)
    return 1j * lam / 2

T = su3_generators()

def su3_exp(A):
    return jax.scipy.linalg.expm(A)

# Random algebra element
a = jnp.array([0.1, -0.2, 0.05, 0.01, 0.03, -0.05, -0.04, 0.02], dtype=jnp.float32)
A = jnp.einsum("a,aij->ij", a, T)
U = su3_exp(A)

# Unitarity test
print("Max(U†U - I) =", float(jnp.max(jnp.abs(U.conj().T @ U - jnp.eye(3)))))


Max(U†U - I) = 1.7881393432617188e-07


In [5]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
from jax import lax

jax.config.update("jax_enable_x64", False)

# --- SU(3) core ---
def su3_generators():
    lam=[]
    lam.append(jnp.array([[0,1,0],[1,0,0],[0,0,0]],jnp.complex64))
    lam.append(jnp.array([[0,-1j,0],[1j,0,0],[0,0,0]],jnp.complex64))
    lam.append(jnp.array([[1,0,0],[0,-1,0],[0,0,0]],jnp.complex64))
    lam.append(jnp.array([[0,0,1],[0,0,0],[1,0,0]],jnp.complex64))
    lam.append(jnp.array([[0,0,-1j],[0,0,0],[1j,0,0]],jnp.complex64))
    lam.append(jnp.array([[0,0,0],[0,0,1],[0,1,0]],jnp.complex64))
    lam.append(jnp.array([[0,0,0],[0,0,-1j],[0,1j,0]],jnp.complex64))
    lam.append(jnp.array([[1,0,0],[0,1,0],[0,0,-2]],jnp.complex64)/jnp.sqrt(3))
    lam=jnp.stack(lam)
    return 1j*lam/2

T = su3_generators()

def su3_alg(a):
    return jnp.einsum("a,aij->ij", a, T)

def su3_exp(A):
    return jax.scipy.linalg.expm(A)

def build_links(theta):
    flat = theta.reshape(-1, 8)
    A = jax.vmap(su3_alg)(flat)
    U = jax.vmap(su3_exp)(A)
    return U.reshape(2,2,2,2,4,3,3)

def wilson_action(theta):
    U = build_links(theta)
    S = 0.0
    for mu in range(4):
        for nu in range(mu+1,4):
            U1 = U[...,mu,:,:]
            U2 = jnp.roll(U[...,nu,:,:],-1,axis=mu)
            U3 = jnp.swapaxes(jnp.conjugate(jnp.roll(U[...,mu,:,:],-1,axis=nu)),-1,-2)
            U4 = jnp.swapaxes(jnp.conjugate(U[...,nu,:,:]),-1,-2)
            P = U1 @ U2 @ U3 @ U4
            tr = jnp.real(jnp.einsum("...ii->...",P))
            S += jnp.sum(1-tr/3)
    return S

flat = jax.jit(wilson_action)

# Hessian-vector + Lanczos
def hvp(theta, v):
    g = jax.grad(flat)
    _, hv = jax.jvp(g, (theta,), (v,))
    return hv

def lanczos_min(theta, k=12):
    n = theta.shape[0]
    key = jax.random.PRNGKey(0)
    v0 = jax.random.normal(key, (n,))
    v0 /= jnp.linalg.norm(v0)
    def step(carry, _):
        v_prev, v_cur, beta_prev = carry
        w = hvp(theta, v_cur)
        w -= beta_prev * v_prev
        alpha = jnp.dot(w, v_cur)
        w -= alpha * v_cur
        beta = jnp.linalg.norm(w)
        v_next = w/(beta+1e-9)
        return (v_cur,v_next,beta),(alpha,beta)

    (_,_,_), (a,b) = lax.scan(step, (jnp.zeros_like(v0),v0,0.0), None, length=k)
    a = np.array(a)
    b = np.array(b[:-1])
    Tm = np.diag(a) + np.diag(b,1) + np.diag(b,-1)
    return float(np.linalg.eigvalsh(Tm)[0])

# --- SIMPLE SAFE C_W scan ---
L=2
n=(L**4)*4*8
THETAS=[0.01,0.02,0.03]
BATCH=8

results=[]
for θ in THETAS:
    key=jax.random.PRNGKey(int(θ*1000))
    dirs=jax.random.normal(key,(BATCH,n))
    dirs=dirs/jnp.linalg.norm(dirs,axis=1,keepdims=True)

    Cvals=[]
    for i in range(BATCH):
        lam = lanczos_min(θ * dirs[i])
        C = -lam/(θ**2)
        Cvals.append(float(C))

    print("θ =",θ,"  best C_W =",max(Cvals))
    results.append((θ,max(Cvals)))

df = pd.DataFrame(results, columns=["theta","C_W"])
df.to_csv("/mnt/data/CW_safe_test.csv", index=False)
print("Saved /mnt/data/CW_safe_test.csv")


/usr/local/lib/python3.12/dist-packages/jax/_src/lax/lax.py:5473: ComplexWarning: Casting complex values to real discards the imaginary part
  x_bar = _convert_element_type(x_bar, x.aval.dtype, x.aval.weak_type)


θ = 0.01   best C_W = 4.631539050024003
θ = 0.02   best C_W = 2.099243283737451
θ = 0.03   best C_W = 1.4726141105509467


OSError: Cannot save file into a non-existent directory: '/mnt/data'